# Đề 12 – Kaggle T4 v5: hybrid DINOv3 + SAM 3 (sam3.pt, fp16/bf16 autocast, class-vocabulary prompts, masked-crop region vectors) + EmbeddingGemma 2 baseline; 70 + 10 video. Built with DINOv3.
Trọng số offline từ dataset private `holilihu/de7-weights`. Toàn bộ kết quả → 1 file `/kaggle/working/de12_results.json`; dữ liệu làm việc ở /tmp.

In [ ]:
import os, sys, io, base64, tarfile, json, time, subprocess, traceback, glob
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', DE12_FP16='1')
OUT='/kaggle/working/de12_results.json'; W0='/tmp/de12'; os.makedirs(W0, exist_ok=True); os.chdir(W0)
R={'started':time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()), 'precision':'DINOv3 fp16 autocast; SAM 3 autocast fp16 (bf16 fallback)', 'stages':{}, 'errors':[]}
def save():
    json.dump(R, open(OUT+'.tmp','w'), indent=1); os.replace(OUT+'.tmp', OUT)
def run(name, cmd, timeout=3000):
    t=time.time()
    try:
        p=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, cwd=W0)
        R['stages'][name]={'rc':p.returncode,'sec':round(time.time()-t,1),'tail':(p.stdout+p.stderr)[-3000:]}
        if p.returncode: R['errors'].append(name)
    except subprocess.TimeoutExpired as e:
        R['stages'][name]={'rc':'timeout','sec':round(time.time()-t,1)}; R['errors'].append(name+':timeout')
    save()
CODE = 'H4sIABiF32oC/+1963rbRpJofvMpepDsGpBIkOBNMhVm41sSn0liRXKSmY/hYkASJGGRAASAFLWy9jtPcx7sPMmpSzduJC1PxvHsGVP5YpJAX6urq6uq62LWzfrX587mO9eZuNFnf8hfg//2fTYarVb2HZ9bjaZlfSY2n32Ev1WcOBF0/9mn+dc8EcvEW7p96+Rxq9GymtapaXUeN9vtymeHv3/9v3XsOtF4Xv8j+8BNfdLp4Kd10mnkP9M9b3Wa3RP4r9Ft4v4/abU/E52Puf9HwbvJzUPv/z/9Mw/0/0D/Jf1vnnROu1bH7Jy0T0+sA/3/lOi/bXu+l9i2Gd7+Ifu/224/SP+73Y7VaeH+bzWarc9E42Pu/wP9/yfS//Y2/W8e6P9Hof+nRf6/DfT/cbNz2j4IAJ8U/R8HkftH0P6H6X+z3Thppvx/6wT3f7PZ7hzo/8f40zTtF2/iBsJbOjO3HozeuONEMFL0xDRylm4sal8JfbpaLGr0WxyLyJ15gW8IdzlyJxPPn1GZb568vLzEL7E7W7p+EpvQesVbhkGUiDdx4FdFEFfFbBGMqgKJjno3Xjerwl8tw1vhxMIPq2LqeDEUTQIYRmUaBUtx/vJ7IYu/xKHy0yRy/HgaREs3itXrJ6sk+CGYuIsqfT2PgrEbx0HEP6ly+qxS+eXl8xev7Bd/eS36QtfMZdjWqkIznbVHn8tgzZ9X/HnjjpaaUal8LmqlvxRYpb/KxJ0KZ+7Ec30aGb2KgL8ZdAaTNiM39v7L1fHreJ08CxZBBIWq9O7Zq+9fXdhPv71ofnvx5K9GVeinVXEKn56fuFEYLJwE1qCPRV/++PrFhf3k4sUTg5qP3GQV+UKfia/EzFy6jq8bhjmFGokLXys0pNhZhgvX5lHroZPMAexh3LfMRlVM3Mkq7DfkcGEZf/Y9BDPXggU3uchXjZ6Ir7xQODx9cTMPYld027WRlwjnO5i18GJx4yVzzxd/ozp/E/AuFsFUJHNXLJw4EVdumHAD1J8OGOmIURBFwY07EbTSz73Zz6Fhir967mISCz2xY3dcRbQwaUkNQjasTS32xY+Bz62NnVCCmzD9mRMCdFyasXEm1jBnfO2E5sxNaCmePTm3zy9endvfnF8aIohEs2M2qKk4cQE5PSi/dDa6VRVRsPInOrVRR+gZsD5c9GbuLVzxOlq5DEP8C64AxJHsLQKOSzfSd95U+EECRXpiBK+u8i888W/Utej3RSNrDv/m0FqKXFiWQCzcBaxCCoFcU/za8ScMJlgb7BVL0kPALV2fiz/16bVhxqsloI74ss8Vi31Tk+K4L6wzMQ78xPNXxf7kSswLD29xAaFenUBf5d1s4ho7UeTcPrQZLr59auSgRt2rZQaYLlwndvXdG5Sp1p4dOou8iS1L6N5yVhULdw1g7OtAm1qwrMHajRYObAqz2cn2xXM3AYIZREAaXVf2IMIoCIPYWcQ9VSuELSOWq0Xi1eKxA4iB3ZnigjZqLBYeAAo2hL5pVG8b1Y1VvbUyhP61Kr4DMMKgTCQXZ7AzNi4i7WBI72FjCh+WTo44W6TxDUAPMeRXgLaPrcDHmQjhcYiPxzfiSI2QSqa/0iaw7QjbBkI7c3XfKKIAvh7vf41/NFgTQODCTtFx3zSIhunjo/FNLbzBLZN7Gh2N57VwTk89X/+Vn+rjY8uA8seyPLz5Tr6J6M38GOsYBfpHPVcqY0DDWLz69ft184LXN129VyHSUGchAhhdbR2MnZE8An04HrJ1FPq35z9Ds+NgCQcbbAUgRLMgmC3gxLxZrJu1EWBdDSjKeG51a64fw8G4cNMFRPxSArYeu4spnHZA6/raw21oeAiGV/1TpMlrbwyVxuFKy8F57zn4ClvNnX70+5sgekUTZMSF2acN4cBMJJXFirQ17RBgGjme7050HDoQTiq+VMW3291dz0wCnSdimO7aWeQoIDUIFJI+oQw0jVNXE6dyXxNHYPqBPYtS8snghW21SMELeyUHog00xbPTic2J+7TBGU1sOBNhmnFfCxONxqcGkA1tuqQzhKdsUhs28z5upIfexl3YMJcVNLwZaPnf2tAYNIZpO8D3wCkL228CrWGjZjx3QtjRU9dJYvUMuAJ8qmPpIyycDQSQMxtHhqkI5omHdEinloBwe7Nl4AGECt1LuiHrw89yxSqNACsJ8bkYb8a3N3Pc3qEDU53U4uuVE7lA7IMIzuBBo2pljV/KU1GRKUCRKE+lFLl4g+3B0E1cXIb1lWF6PgwD6EQSIC3US3RkvAHydFsVI4DGCCmXTlMZvBkCybo00lqFSkBLBRBTAdRU3FpydPB7vKmNburNjOyMb2ujOT8AuqLGDzDAsseqbP6Vha9uj6lW+YTdWLVNAxgvq0sn6q1Vu+WfPYRHSgiRdm0azM7pt+rLxlJPLCN3zEmCFiE523GwKVTcPtmY9l16yOo99/yMGU5J4GvcUGsvhh3LZ5h7vfIAeYF/xxPp+csfX61bv3ivi3wz7jkHjrOYJgUgESCybeB/4O2APQTxQR7sP7qJQAa0HieTd9FDbKMP9XMLT6hBbfcFn3uMt9AaPPFDk/mFQcNsn3aA7TLbnS59NLpDRZ+g11LZZvMxFoKu+KMzfAcRQVqxm04UiQt0AGLk+Eof6NhXzL15SxPYIjhQE10DtgWIi+T39XRq1WyWhmKFnr78/uWPL55cAN9VB9YTeE9Ry6aOD9Op4X7ylriheKzDLZS5K5KjHotUJs9F35h0cMAh5+JGaFUF7BXE9klyG7p9LjtdBE7Sahr36ih9IdEtxaHxwgt7dIY6Xh1/1NZekjvPWk3xFpZwBm+aPXVwyt/FYw8x6K0AiTJYQ8mpM3ZHQXBV5we1eAnLwxICPWkJ1wRJpFishZ3H0JY6emqL9cTqnj5eIrc7cxJ30hO+6wIB++4bAMeV64NEuwDqA0eucMZjkEUcf5wTKPYe3++asZYe2Mhcl9Ea6yPewMdZ/sDjKihz6Np4NXE0pCi8DPjT9GLbWTvewgHWQDeYzWeGIC8vaKlcPnOXS2zEp67MBchTkU4yjebOmvQiiE0UhUwcPBbS4yTis9pQ5Xt4GLxQbX6LbYqm0ImfXQYTByXt0BnP3VrTbBhi7TkwJ5BSAYq1PGdSZB2Rb1HF7F0MzKV8+Tp7V2iBzw44OQBwWpxoOZ5kR1WaU7ooCuZA2FFVYF/dONEs7t9phPnpPpG4f0+HGW6o4jJiV6m6YT/DkzIUu1keOQPEXlpvXX3dsWrpGxBWQZyPdHmWA6GZesDTaDwdmgT80gyFJOuFlkeRXM+q6+KRm0S3vYwl3Nac7J5toQl3g3sJqBqu5gsQ5qNcizvOJJgi4BmfOMh7y6OG92qSnVNw+ke0bQPi8bNOcaalQb/PeN+fqeTzYBT3gRz29oMTcLEITEmMc2eDXDbAUVgvfXCnUeOAd/sPDTqy8Ig07rcJP4wK6Y5N5+gIBukD0jsL+GVnSro+qiNS6q5J5M7RjmCVbDNsXibfNVAq9iUTbSAkSozaLk4bPwZezzse4TB3HaZnVO/uqifWhe1C3YMAsMYRbEwvcZexDudQieN6EJcFKaDklnYAK8YOcItMYHecdRawEUAKgcRO+npGnKFtJsnI1gHVdP21F4Gcg2oj7fkLq2l/c251cddZmkHF4XOHQDxPOfCHRAcTFSj23IPD1reBwUjcrcb8yEVl4j5q4K+WpNSIEzey6ayLYYANoiWN3G6bO7FoC1WST8V4q7MpiWR0GDmJPpgPetAWrCl9seAUxdH0hqxxtAx4U7NoTz/7/hLeEudGx2ORTOCu3dmXnA1MzmaZC2WUVYTqyncDbltNxdo1IB9+TKe7PpXaZfM1YSKccdgjIG4QLABWsBPCVVJoBx4p7l0SDJBxV/6YVQhmuuH0KaORjjvNW/YBBiYc0bphkopb3+brgS7AigFMAesSV4eODCAUiJZ6bpPu4vwB6d3NboXWaOUtJjYV0Neo+LQnHnDl0Dh/oYPiQR4m1QhL1VhfQ80VvFgEs34YgahSVhbDzlg6VyBVRrGeduZuALHs4Iop0BlKLABtxUjqNBYGS+ws88pb5GbgUYvOvCivvkl5CHmFZWIxpcFLWYgnP7Sk0gfFvRjGZY+jIDyT/eTe6zAsVPjgbsrpAnEE6ivuaXx3zCPaVrJe2NAgkjKN5FMg5w2mVkUcl93kVVJSKdIHwDDpK3VLKiLZpf7Q+HJDY6Cu3THOH4gEHAyI+Q0i81X6v8GXMSb+Iw8/pLtEdGFnAL+sT+nJlPiORTAy8R9dMY5vAhCNcximHSHXAcOaKs4FSOUkRhKsp7ctecKIbSeks6EuC/cSa0JBiWNlrQCAEaeFc4HKOJ07jQYCgN9ia9ewHTU4mKXOPmGtrobnBjzUqD8k3rBy8Bthdz8snzS4cDtIFYx/RFoSeK0XVU9b41UkBHUJ+EAfGYB4S/U4m8By0BgO5K+hHDo/S+i3Gjgvfjby0f2WSgJAumfUAL0rHPl/eaEOpWjwVSzPRw/umPh9ZpPbWqw8p5Y/yMxsGE02O9LzjAxkyGmzDeROG2YXAeoPkV51DttKpwHTkGAfYPHlGWyGrN5fWJDPU2JsAipMkCvK7RGQx5NGulGI8wAY6hkOZWuSm0OeY5zEBbWEx4xWFfggaMiFc8KNsH8cKW0lOOIIKkPc4/htmHJxQIK77aL8RzzaJDb4vmj7bsabIL9Fd6zmSzwgXj7/wQn13INvFk7y8lz/C+snB9YQ4QbVTGcysXEn29C+/pcB/Asjwb6y3Uyt3ETArcnTB+rBLtbu5GlwX7/DGdybVFKOHLmbmCgnnQWIEXyFq8H2h19EBCQ08XekiLpGlAGe0GdVbmQsgoROW9PtDP5EoBA84SmNCy8QU3IwAXqAujdZ3Q6BB8i/96d1LNKU3KfcVDxscxVOcLXgiU3Y2C+iJm0ofkPtUhd9brdYsi6vFf2pwXoYrAgMwKRPWxK+AEinAT8HarMMk7hPWwLfyweSv8ALd3MCHId+d3RE40TpEACAsIWP+yppbPT8wuBzEysi6t5osiU46nVqIJWCed47ORI+jPdqIumtUh3tUKzwyjDPILmDnQoUlJhphggUXU0krZ2fipFeVuC+l1zlZMCwGBabZb4k1afCpkx5lLQa4eewJMTzlkIRhvEfb3cl+hfHdXeVIj5Tj/ciHYhw6kQjXire3+x9JQUuL0Z2J8JcXz/t7apvNeiCye83Gw3kECLgsmdOSDCvimvYPmXwaxqBQBqHiLfq0vMtyOIwuvROEyRGOBJSWxA4m4niV9HeJqkCUa6GrnNlJ9UYbY+qsAPuh6nKDf+uAaA4AJw7f+Yuq4nBUUum0/EPyzxy6cL5LhMRkdIwXeE598RAfhtmwMaH8uuQTppkniuYlcOXuBxZeVqdYXX7gOS6x6p0qSVZDUqtYndnV6Wi9wOEeVEuV+fO4CpDJJ4uAIx+KcwcFk/w51XxUu0DeDug88SUuHLNVx2IE9VyET8JEmdhGFtXwEBa3ig24jldnLyEf98p+8LmG7wZnokr9xbvc+Y5hmBOjMBOIQ5LkyDn81pj3+Ir+j6Ad0PotZf9woZZFENlxZzIbL7hGd3qpUiPHeR+lLHNMjtK9w5kAI+lYVmQ49oK5XUcSBV7MQa9qyGb3ewtkmrTf8ASInFRgnEWi1vhTN44Y7yPmaPdDJJ6V/BeEm8nydsv+1gdb42CdLedqW+CdhdffmF9/p3us9Gtvc42DC4lctwJWhuhostQgFa6lx7VAFxJ5AUQFh8MjfRaCyZDFeWp7s5KdgpQfB5Tq9iOajWF4zw2UeRAWWy8ivKCYE5QwA6wiXncK3O6WAnVMwmwaPB9oLnENH3ZRxBvo2OuCDB4uxAOsYtKEdw0wC74lR74+KwPw2FK1oexAYb1R8Z76Dd4sD0CkYIePDB2jZHol+Ke10iEkIaikhq+u8Qs4zcehfzB4+0hsFLBIHdk7ehUHe0s8mGRKu63/sJZjiaO2PREbZPCwThYBx/s/w/2//9q9v/NxuO2edJ+3Gg3Dg5gn5L9f+Qiv/7HeAA84P/VanROlP0/OgCg/X+30T3Y/38k+3/WQMPxX0MUQENVNM0OwtqfU3ayR6yX64BgPwb2ykPuR73Eq4EaCuACuVPkQVgyrOKllJAOBaklZUVHDXkLr26/Pf/5jFTt+OPZ+c9o+4+XM5Oa5xtS/Jd9or6cr/DmriD+ly9Ooypxe5HLXC69lXyvE1dQmRJE3szz0UIBfpF+0h4HsQcSbU5SdWvM9GTC6jGq+9DMPefBgL4LSCqrW/4KezwUiN8nRQABROrI2dI+kUzvHuP0rChywmgnWLZNP391af9w+eIZNCWOhAX7CEqSgXnBvvysZBedY/TebXi90+6a1BBXuYuPiv3sybPvXpAIgRd9upRLmBU10BkkXfozXkxgKVlz4CkDWxcxYOxCpUhd+yvYIUbqsVQaVUXuloG0GbJ+pPQaJLkD12p7qNDAL8EqIR1HWZNBNyA5le7Ze6kbeBw5lUNeyklfS0maVSg4MB6RHMQ19OWj8o78DjI1cswS+6xwvQUlcgi078YlToVng35IQWBYUApDYzAjHNQOjbCUwuOcFM76yVxreGFSaHHuxHTpnK2CBu1oICSqJya3C/9WdgvxjD49+amEdlUdfRq2jFdzRTNAHvdJv0tltnwq6OmOSb/ILxqt6TRSdzLZpQ7VJoNCb4n9vxBfi+tBAzpHU1y01cQXphPNkN4UIYRv0Ej1K1yVVIKMi/Kj1FBwUYNFSL7mkDauRlXuBHfC96cp1rGw/A4hLS/XQflBj3YELOWdxk1KJXf5cgOlSbVvUZXOYL4/sIQH+e8g//0Ly3+t9uOu2bA6VrN5ctjsn5D8lzee+dBS4LvlP8tqtTss/500mm2O/9E9aRzkv48k/7FAVnZdJJe3M5LLboLoin14xWiKbi3SghJEqHVbTD3krio18a3ru5E3FnwNLTQW/TS+cYjFfzfMRrMG/5yClAAFpx7x/cjucC/YfjCdemPPWYgkWo2vzDfhrCc0+q6JhnnabVWzdqGpk8cG3r3FgbhxxQrN95HLI6tjcuZbLZzoltxYnvzy8rL+7NWzV7WFd+UKvo32yWFbf/7imyc/f/8a5aofzl9f4qVHnMB5aMKkMo8XtKdLeJRkXjlhu/U5emsFi0kfhiN0FIHlzQiMrwMMIDrSLG7FhNhU5k/PcLRTaA3kHWoPJE2bJdhgFcXuYu3G3Lc7JoPnHsrKKdjFNIRF0MniFNbOfv76r+cv+rwy2KbvJN4ajexCkCShcKuJd9K4ouQ5ju6OVBitYUwEfa+4rGYltXtGcyCU4ZR/6uZ2c8uTkFfqmZWQQCDEJIIv0XoQ9QijIFgIfqt/t/mVRWZmrdlgSBoRlYRsZNTdTbLwRu8lZZeWj/z4QZSUVgyO7y2dBX6bBDP8mAcAYvwyDm7IrsiLJvTTifjn+HZMfpfaMkA7T/VrBFw6fQJoNLIsmXv4ylBuMTlbwr3mDcpmg2VD5dmZrj8gUavgN4Jeue74KgSBLyF1QPk2XoKCAyUUXSuAopvsDUCmoG6kCrNlKFF8NqulUvtqm7mCmVG+sq+Ed1kwhaKFxodyaClBANUbpSfQetkeGxfDfvbn89da5goGxAmtA3YWvXx98fLZa1zXRmq8nYmeobuv3tPzF7mhEsCg6G4I63kPXrSgJcE+WNrffdP/BkgurPbRkX6nQXc0L7yzC917hBiOgGBzd2+ULE9Sq6Ci4QPeLxbBVDKevEqNucmEplQY9WWhvQCCQFEm5JBvXG82T2I78Be3JIpCMSinrHqpTSDbBB62kwEEvireacYTMpQxIzdcOGNX1ybShd4khxU0w+qJdcH0H5qUN8WkPjDp+pMNSnO1S44BqDthBCYIkfm8jb6uejypCsYGBrux7VVUAKoGgEl2mZSWYGaQNRoPxwahXFmdxRN8s/TiGIiifBaZ8jcVxPcr392EUNedpEWyR7LUTkPMnNGJapNtaNF4rdDNoNcZFnrKFyx1hmWLACVrb0Z7wt2eVi0BC00lAf/H6MQ92emlITchKm2LVQcpfIYwGMcDVL9Y+SiZkOOQPuV+ZX3qXsga/btSU4/ki0fD+x17k3HiQa8sJHVoIJ4ncLq0SdzyIdvHEORck/y1He4jIvLk0raHgIcF7tMVrJKufiMxpfbMOFx40EiVDcS4DyITZZZGEUHlXk9klL3r08OnRFbSk72aeyB5OWU8nh1z42RDAH3oaFKIkHrUAF3pKRXVloMOEZ28f85IOuhkkJrsIulF3kgjK1Krq5EvSsYVeewmALgChw8OIgzIEWaLTRI6jQERjngbpElrL4G9LqbjJsLHpfgb6/aWa8d7+BypOVF4loSggh1LI/6CX1LmmhY5N3mzPkaNkpWes5HWP8dSeYo0FbXZkkPXV6h6H93KWxXgU275zDAKdng715H4VlpJWnpDedPABnQj2gi4V8p+/GSdl24vvNTg85EYzTMRbL1FPlAaukr07/NHlUxQYQ8muU2DmCCVn7E2VF442y7ReBWk7GNYzS11sCarcJUNdSwZkgY6wJC3gG/PcIujnx7GHonFV321oVSwnPt3xKToyUsXjlj0wFbZip7wB8B8y5IvTO0GJbXZtlx6v1XatUAl1wPy3ZBWg0FqA62WLQ3qUN21qMW3RKe04R5PBTbjSheqGIcBL/WIuuKCSpdDYjpGGPSh4Ku1T93ewNuBKxfjM5V9J2P2hGDnpZj8qGNyfBU//nApHBB/OMwM0XX9ZfCz+Aqw7QQZZxiUh44IW94cgFO67QUrjA9yhab5aN6GlTIrUBgLMFH4kZ/nqDihnLMAVUDKw4dEOQRVWdSDaV4NmmhsWXRsI+4MXpgTD3YCttfiDUQPi0ULgx3u6Co7aq4wHMme8nJPDxAQ5TJ0h0iQctAkkZHD22SRQNDQ1YGhwVv4F28+8IVDBqwjXFZ0ergtl2/R21auvEVPpJMEXgptxBFUpP5WKI9iJzVs1ziiBmoOrdux0LHj2ojfYKM1agda9vJ3Nxgta4XQXSF+pERJWpRu+d4EG3Z2q6KWYdHXLfQcttCt1Wq0jczS9BneqycBqWCIuJAbZYw+gjGwM3QuoHYI2xKSX6cDQ4X2qJFL5yxyblFLoDy3QeqHNkBESG//8dB04/RUcaTzSxonY1byeTbgN2LJmXAG/429K3dI1CgYeEWIE3vXZbZjyJs89A8/qHn/h9//HOK//9Pufwrx37un3UbXfNw+tTqnh13zKd3/SM3OH2IA+ID9X6dtdWX895OOxfZ/J92Tw/3PR7r/eTWdYsgupdxj/e+fndkMuNSJkzjA4QNv6gcYLmkS3PioaMGbg8tViFILSgg1eFeDExpqTANUNUMFjkFBLnjA6ByZsTN1ZdQPQzj+7c0chBYBwpcbifoVdVf3fAq6UBMYqxbPeJcvaSbi0Zeke/nKtr/Ex189EjrFfpqDGLDw5qv6xD2pyRnQlUXk1pyYYyhO2DdGdYLXWZ4/q9/UZZt1jlh0uwQwXMXYPelVcvrTGPrHcXxlh06U2D/+CN0fi0BFj1Qv47nT7HTNZJM8ojHkXIn3j4GqmmFSUeaIqKNkdkwGuaqTNt6yUQ0pf6jvGLYB2at5zG4vGFA3mM1cDO96iwErclcrHHcZL1g4FjNGjaU7lni+SrwFGhkxQznxgPcGbkyn5Yj7WmF5UA0Mo8+epnMpRoWgO75bhAmrh5nr5DXtKy+TMBM6s8AC5NmIPd/Xj47qRxqObLyKYm/tclN5b0gvxib1UAo2HMHmTuPpo2sd+rfKspG7wE8dY/hSB8a90NO3M8BzDN8RGvXm0VGz0TOt6b344amhZYOk4Q8zb6kFRdBaJQPV4bBH7rIa8NcADWMrIgYCqxwOgwp9zji/heeVgmRO/WdS7oh1X0WtdFiMBWbbFDtjVJSNl2woOlIqRCiFmvczjkyp3eE47+t3y3vtrDD+yc7Bl4MYb3mqTqCt6T2aBsI7udNwGXKvJBS2th49x20XFx3VdsFjSfaSJinp9EjTzWODd6z+2+TY+EKr7oJWAVzLHveVd3FL7S4BArIOyFfmLApWoW5hPNqcB1yYRdBAYFGUqaoIY46iiS1vOb1NSEWTAR3r3ANF0CrvhivUK2k8+A4dnaEn6Fao3Yw0Ay9Pb3aHfwhzIT7CGBaHKQEJXwhZEN/YtRr96iNoi2KXWuLLL0WzXYgahbuMhk2EyY6cG3Kpg1Fkkv0cxb7CLp+oyUrKeVQMY4A1kKpllKQ0W8cn5S+OEF6iEC2tniVWYxxTunII9TSW3Rn5oEraZ3K/pcChZShGEorTPUE0FqRzgGWNdNYL9cSUB6JABTK5hsg/V5afUMcoBxfKYMijgi1GQNTn5tzdTLyZi/ou1KrgxGEiSGhUHVw7rCRgX95RY4/KjWXXIfNpRoHvtpH7YZqcO9t3UOfjYj3G6mKlbZ0gLvie2DK7tp9WYCg0w5BRT0KPLiGlfu7oCkM9bmALc6xM8peG2UstGkXbmKTx9bAEPbrCPZ1Fz8FzVK4bn8h97IWj9K1bZAtAESc1CqtVeEchoPCbPdpVaYSVqnSKy9f4Na3VzNdpqnAMNMFBWCKCFJ4mC7Sja0hAMGNBmMyh5hBWZVC8VcVpFa5V08patouNoQLBQMu4EMJLBmxuHDAwCkakDOAjqjEw7eF/WG+p8lsKGRku3M1uUqyWYgv+3PmDfb+jSQ7Mh1YNGHEu37zUIEEvn6DIe7D/Pdj/Zva/zW7n8WOzeQrfWwcF0KfwF4+BO0ri/yH5X7udRvuE8r+eWIf8rwf9/4H+f2T9f6dttczOafOk8/hA/j8l+o9x/SYeyMoOxcv7oPcAD/n/A85J/X+3e9Luwv5vdw75Xz+a/v97B+NNjVx/PF86EQbhRtM39VPoVoP9ruOquF65kQey5gpeYkDwCQr77tqNbtnDQjRhZcURhVg/ErEL5dGxG23q0G7xxq9kOMaNCr3bIF35ierFFM+zMnGwisYuJ8giHbrs5/LVzxfPXlyaywleNIToSpIEKolDXMDk5cQoOxiwJlypvcmbn1XikVuJozEFSa6K5xSU30mc+sRZe7FNICFHgtIze4EQTN/QCLVKXmvKMRbv6zxDbUuHuiO6LlSBoagq9SPKTCjVfVJHm2uyfrcl+k4NqbstqAwNaSyvdLDqvTOKSTWOgScnPKBrCi6nyTXH6cmv9tyJJtpweyx317+rSzlTrJ31/mY3ONQQjqQeqaCz1N/Q0hkVwLxYaQah5vP7OtmHXLpA6RBF64SheFGjlVWGZwL6WU6KtTN0S8ufIbaxwVfOe/9aKqPX2yuU4aR9F9/Tclb263bXRikMgFTXFSACI/tf5y++pbnF9fZpI6zfoYIO/aYo7OVNlA8u8WtEOsp1tfzInuI2G+tHGoxqjYuABkP6aQf+hVZzGvJM5xSVYq8tZU/ekuATUu8cBlbHTAVTzGzA0WR7TQpjq0PTVQGdSCvLXEJMb1noPusfmizGspAaf6Vxmmq/HYk7eODGY0yhFRv3v8XHvw2+B8T7bfibrg/+0xgeG78ZaLuO68zrqJT3Uw2Tz5TXSbwV8A3/zZT+dFfA434Uu+4WWXoE5TWjQjikwawwV1J9D32SwVYlsLTP8wSQkUfoOfJKuGv85j/xAW/IPSPuiWfPxNO/ijb6oHHRIJo5PsAywqvabyiobU/mipujPw6HRZEEVnUjyexIkXP9r8Hq9Wrk1s9dNAyrI86sg/r//d//h2IS0DS80apQhzIGurwg5Ok2CQjB88UVqf/N/81/Kw+Ct9l58XZrfG+hXK1WS///zdfg5NHgg1XEAFlDPjAq7KiAdo2F3ZLfjJKcGqSGJZr8iVqJHfR/B/3fIf/7Qf7LZQv54DZgD8h/VrNZzv/e6jYP9l8f5U9KRSDAhE6EHqjxbYxXc2f4RVrYAMsSJWj/Xr6I3fdbcfa2jVeTtk2JeQu5WsgtqeCOTKhXoWhoajDmk2i2wnhs5/iL7u6dkDIQOPKFrtVqqUwVYebKSMVI2lk2IPOlYsHKjnLKd1VagDyQJWd3XxgguirIz4t8VrLWLHQM3FFDZTfI9SuT7YznAWZH7Q80P/BJ0pQvZF6YqkySU83npxnuHhfnS5Ajk/l7uDdY4bm7CPsa522fO0sMgSdSj8aqaPSD6ZTmi8sUmrRM2HoM7Hg+4ZBjKpWBY5Iw7ZjSd9IxKZWDY6YZHDh9kGNyipfDqXPg/w783z+L/zvE//1E+T90gV85iftHOAA8wP+dNJrl+L8t62D//9H0/1KlSgrVnsilIuU4INqMzKhVAoKqTD5QpcQD98P7oSkuONULRQmS+TSyWLpm5UkalNfDYLsLwDX8PhVeItDOfOGEMXAU375mz3u8ftCP6zWRBAuDVEe3mEScfAgnQqcgslBWPQgoV3PlwiVnBHHhotP113dWtVO1GvdCd/zbXKc+RzZGA94n519blPPG8VXWogVa649v8/cFGWfMVwSSP1ahgHPxif4glrkc8OhM7OWkVXal38NGE9/2flx0diPwfpx0gaPl2NDM/FUpXu/uTmDx97DPDbOzn7vf/QYd/p3IdfYz5Ir1zbutqtsujtGFPrDYBka3ppyf4ktgjb14T5ccXjU/c031gSw62UP2VNBtQspmg3sqR2LTHpB7sg72MOY/4TXFtUjvdEp5sxxTTpQDH1xzCA2cKoYdHnO+2C+Rg5dQHJ6JS8qxzdgGDRDyYP4lEItyLgn7kDzX45kMc3B3XxQQGXwKsflXBQNfq5wwaMKrSqV5L6WPyb79kc+beSZka/mHFXdRbhfWyZtg+hTxOa4WBpnjgHkcUiNOoH26mES1Oqdj1bPMTXS16QfC3SSRQxXHgbSJt58W/RkoqsglJUYqpJae51P/5ZNC9aCJgpvCfCt681ymVGyVnBQ44kGaEEnaFVMUM/u8V4oKXUjBsyOuxRST50ojWhgRYg+HEXBvqVOqK4FtnwM+7gRya9fiPZDSNV3DQhrXyk+/VMVPr7MgNSni/yRDACwxZTJ7jNMOKFibIw5XxfVAHoNDckbPO6efiaQcO/ynXxRkL3PBwfHF6xTkxQjLfMmJ1BEHxtJxnIvnwyO9QBSxejSRDn9YDfy8PxNPzjk8yBmeWvLb+Z+L2ZauvSrPPEsp+VPuYrE0jzzWqeUpxRKhpEA23blRaPZLlOSVqM8B63BJVNhzjMGeRl//6ZfBtTcsZWvfah5Xc1cE9b3NwOwVjH96ja84eH8R2KowRrzBY2tBMUwofEg5Ysd2GHZ1z8hG52muzlkRsIAxM4y5ToyNim9OEXbQh2WW7k0Ks5H9olBYA8lUcaYqlZLqGGALx6Aqwg+/ogKqeI2LFOzZs9AYizQ9az6YPaAlDgsPFH1ZvNolR5aLnrgYXA3TO1EKuAJcFJC1BeYxy9/HJpTImEKLNwj/tmsVZw4VinOfFeZOr1XA+QIwcHgEcQXn3CgoL1pfNPCcROTE0BiFDPaw5MXFgqkYW8nDoh63hBlhuSn4Rk/qApb9WFhZl7D71FSxYJ1ig+CVoxoeYqyhHAxiTt+Hu/no6G6qXXx9d3VfCj7kh5w3fY20umUUXDUulKPGPQaSI8Z1T20YF9fHvLpBaNkETM+3v329p8b5n9MKy5jsK27TkphW4qhYHLYblrc4GpJiBTli3U8y+BFfwPK+TSdvVLbISp7WF4965KjtKDoTPro4DrDxfGweeGVyHgCTc77r0k8Fe0sj22OOUvIWudNWvgdDtVXuCTVgH4P8cOAhm3IbYsK01bLwHDO2qnopYGQhXHfO2Cpbk2lbtbQjTparsoFjNSvzM4LJYW7fAtSKo6/umpGEJLC7vXyWV85qC71cmhM0VlDJXimVNeWTJCsECrcWq+yvUj2L9gecfNUHdvighD3ofz+q/vdg//1P0//uiv/SPcR/+dT0v7Nwxcasf4AC+CH771arU4r/0mo1rIP+9yPpfzG2PLB28SpyOfmZLoO/nAP7BxzO6zbI9zK2G2dGl7fCwWo2D5F7YIVImthLKrBI58Gasjh0Qe6qUHK4Xy6gNAZcyUUBR83S0l0G0a3tLCjoNMgngoxTUWPM1pMZihLTIy5cZ4H61xGGm0G9XV5ty8pa1tFKc+9UcUtRNj+aeYNKIF8VHGyZc3nFFYxOo2J+7oqFjvw1QC3G2P9ahaOp5wrP3MTmOMQ2DQ7Tv5G6QJtRSF2KS6xRBczvRBVtew2wQv2RfV8BISzeZVe8x7y9bDmJ8aErPBXk0L1lxp1Ty/jLxqCnJE/nZ44GyGgCAfLixQBkCMmRI6OOXDT/NCgSDSKMnmmXmPd+AG905MqbR0etBvLilVxptEpNWCCSlYihRgUOai3VQulb0YQB939p2edPXn9HoYSdsTsKgqs6u/LX2Pm/FsIg0TK3tlhPrO7p4yVy1BzvFzuQ8xr02sPCoOJbADF0hWbPu1RLqmJVjOJ+t41F9lSuADRlsAEZj4BlH285IyEmTqWXHJhRtC1rS6R0R4BaQzF7NsLbIFqL+wpG0XYLo3CXYXJrj53xnKfwHhCv/D1qvsoWUKBCWetHyFQMEvDOnACqdBo2nKrAoyyQeKkQo6jEwSLIOHQOFFWr3Pj7FpmkXBQOKeowNmOwtDtlW3taqnevfJZsQC47S65FsXVrqWH5c8gg1QCZ3FusDaMsFW++A09YnE3l01i/yAmYRjGqUCqblj1jznIC7oUUWbXdB4J2kGIP9j8H+e8fsP8B+a/RaIP8d9I9aTcO9t+fkvyHpJj8+pRK9wPKgQ/Zf3e7lpT/Ou1WB+W/TqN5yP/2seS/72DV3ahk9NDjZM3Sl4xfURpuLxbz6cILQ7xIw3Cg8dhZsPttp/FvwKVGGIPTd+NYbBrmCSfoDh3yKwNJ0KlE8CBY1vwAc7w0m+0N/I9ZxddOjDfny2CN1+pj13dVIjDhEP/Bjms0FCMVDpU/KD8nDGbxcEsWRNHvwWxiVf54hSbK9O2FP3f8sVsZ7XQGPhORP+Ng5jwtU17G2/CcRDK0vSjZW0y1u1HmyMoDl96syBat+DK37D+cq0Lutzt9iIu33OqSW87BfJouja5maS49Sq+Tuwznvu6uB4+Im6SoceXw7AYw1tSmDktsyO7MZA7CuA8CrK5bVhMjzjdxRqPstj2Lzw7wMdHaawYCKcrezU6nKvQm+p3SP8iIcpIUgO0KSp7Ky7TRzCRsYh9RrIHB8pfmjTdJ5sDPguzHDfHjOYWE5ec4GLyuzO72syRUErb1sptzXUu7jZ21q6AT3mu8VuoS8O7o6BqqpqZz90Yl41zpimXv2mdIu83CnonMmRE9mz84T/s/g/9rbvN/jQP/91H4v26e/+ucdtst83G32T1k//20+L8QKKETuTadbh/4DuAB/q/btpT+v2N1um2M/2KdHOK/fCz+7yn6bQlHxEtnsWBn/BrqMj23GAiGGKZcHAKy08agE8jjpaEIskgEBuawfZaFQI/F32o11CpRF39L3f1jDo2Oickw2IIOR/c05CDuOYaL1c/Q4qsSn0pcqU7muSNMZHNsdf4NGE4MOzJlJlaODhlQrosWd2iLO0GTYVavCX0KjFaCQ0RXN3j+N6l3g+NfNZQFtIlc8eOr1wIjNsDkkPlF5lQZwsX8k2xpZPNof4WBcthE1J0QcC6Jy01ngnWmK1iD4tCw5Leve6lxPLDJsTs2lM1pIcg8h1Lg0PoZu8zGy2QC5tdIWyvquULptIAXuo13G79nYeP5NoWC5jycnPd3GKLTcufNmnNBdd7l1Vksn+fSdxqmp4i40xWytd+OHIO87K7T2F1J4tPOOsAhP+Q8WiiPYbd3mXiXAt44Js1/K+wNYM/OoDeDHt+lYDNbkgeZCO2NXVRU5GbFMyeBkpwSAJ8cX2fWuXjPk6K2Pl04aGIaTFUmSeKdVcbJ1BRZWvVNi1Z9VBkDHVNa13w6OG8KZcmOErcBNtOjRr1yiPVcKWUv2cvz+TqMDHpAk0DpycoPdA/kDctQjynhZt52OhdkmO7JPEoJhzG0UPohtT/A3kBz2hQz5fgp5AxZWV7Hg7XXW3tkGZoWG56JNV4BUsghxvs7KFOv5wv1Gs0JBx2SUYFwZGc0+MxekuxfXRIiqdMMfm/CPSGIUkzbDkTkXudDETn+Qy3k4tkUmwj9mZaz9kxz0GEKMRImc9JraOTClzs+wKZwzYk2vyhQkjkgHRp0V4OXIXeN+6yP62I0ozchJ/n7rip+RQEyC2SUZamm0hzCZ2sX8FRs0moQRKri2siZ2Yco8zM2OKakF4wVb0IEni/vRrFkLt/ldSp9poLnVNvfpQZP0AjRvd7y6uoxCmEZMsbtIWbAL7S97SkU9wsofj+8LxoPBx75QQC8i5a1eO++iVlLQodOCnc0CIZaO1MSbmJ0Omk3Gj1SA3n+yi0U28C22VhVcQuftxa0vkFlho/35huZxxNepM9u5TMQqDdVEaJfA8rVptURR0LfWDC1TcMg0Tt9eotPbxullInvs9Bw7N7BxO7VWg9kAsHbRi28NXo4pu9w2Mfwq6qyC27g5YZf/opzO4Zfw1KGTiJwgPqYB0+XcPRSOHIGUvEV8GcN3gYeZ0MMk2Ex28B7YE5pDgXkyZyD8lfKG75O/g5g9ytMq70/X/WD2BenuOfeq0Sa5Cmx86QoY2I66yIiTrc2tTfcwr1pLkjZn/JBynppfRmkbLo7RJpsB4hsetLsDMNWPlnrdwSEe+09wrPJo2df92ngtWkuSQfuYdiDavtWtsKp5fRV19UcK5HH74K6coe2Kmsj3t+GpEm772xZ3wWV0jPxnk4+FAfuijG9qNnjR6llyiMDyvKPMy4bXMOTomK7ym9ifBPn+e9PMwLYwf73YP9b0P9ZHYz/fHpyCP/wSen/SFfxh2R/fI/4XyeNTP/Xwv3fbJ8c4j98lL998b/YdvaTCWnATPD7BjTIa5sohEE+QhfbzFWVq7ZGBnVQZk84rqudep7OHlf+eLfffdmRl2KwkuGtcqXNichQDX+UL3VVjC78vJKqh5QZiwaPiLF6NLz/LcFfxKg/GlKOxBo+AG5d/oyFTobe+JRdOdUL47eEsrr3qQX8hm9alP1PdSW9Ae/I9TBvr1hLjJ7ZmN6L5ScbrPVflv872P/90/i/ov2f1T6xzKZ12u02uoc99gnxf9HKRxcKG+/84NCYf+j9/478T1arqfJ/d06skxO6/+22D/zfx/j7/E/1kefXR048r3wuvqGY63Tvu3SSyNtwJm02t+eUC+IMHRWWTnQrMGs4hctQJRCJptiEuQhmFXdDHNzriyc/Xn7z6uKHFxeX9i8vLp6+unz5+q99F+3OKu5aN+6EC9yT0Pp98YUlvmiKL1ramQhvkzl6jm3HpxOSs+PbYW+ysaFaGpVKlK//UrXTF02ewBctQXeG6bC/uLPu4e0XbX7f/OrfLfFWzCI3FLUXQvtP9nd4y/zcW2TmDBjgfWUgatNsEAVvoDr63nNzQ/H2bXk2xWjLQgVz2h67fM7j3d2VkOFqxfv6JQm6XhQIM+VzgzGQCvOmFomN9ClHaqFDTkdaY3uBnSFxz8QkAAbSXYsvfJGzcjsrPBK5yFx4B9XRhE2NQtUBFuuXOh6Kf/93cVdoQ7VIqjxclAneuB3o+oH/O/B/v4P/67RPTs3Tk3bz8cEC8JPj/8jP9wPzfu/D/7W7iv/rnnD+z1bL6h74v4/P/1EquFrsJswD6mlSNgPd7YHvQ9f0noivPGAgyK4HQwGwSRuaxDBHxQnJzfdhAM/E0/5uf/Mcd/W9jUzG72Spvni6g4niFlPO6V3R/XNcEsbeLwyLuZMPOzBu8/cwdXlmjli3nyjLe8Z9pSwTMWc0EcUCM0DsL37aP4+9HHgKTMWBw8xSnls2KAPBSmPIIvtd7Ht7ZNKn/XeOLYXnO0dXHE+xR2YpSUZ58v33z1/9+OJf6GQ88H8H/i/H/7Xbna5pdTudVvvA/31K/B9HzghXJh17H37/v8v/Q+V/QgVgo0P5nzrW4f73o/xNvOkUzr+ZlwiHkKBOfEd94o7hM8IDdbT7eYWP2eZjx5m0XNPsWuPJZNQUFi12pVar7Wuxcnx8vLfVr78WtVajUT0Rx/wBDzgy9evI8WPga5Zu9JzL675v/hBMVgu3FMyVorm6cbBYUfipnDE3m3cnwCK5BRvvbUvFqeskNhr6UbTjtK16XVbfrjEOgmgS2/Oq+nZDVtuLqWljtCZ+qFdquywi096q+a8c4KnPUYQqx393RZ1rIjhsm2/UbVuXUaGMXVGnOKWpBpRAM7ZnuOMRzW8cLENuAWYZTTgWEExe3wbJezTBMdUV7PVdMzQq+xA3DGIPF8oGUSLASGVbKLyrhETmaet03G2OTPP0pOu6LXcvMu9sYwutd5ZCBO80Eb/p3xS9z2XhFyrE2qXnu/sxHP90EAMQbqvEtYso2mpWxTteli1zi9bBbHCL0Pf8fCv4KO5t42/i+nEQxWl8sv9yoyDWMSquhUlZd6CysY3Lf08j/xha78NAmPUNSEm6HMmOYp+L6SpK5m4ES6ZoyQSEP4xenKT+WIT7+hs4WNAvbOSK2Jm6e9EeSw9wdkNFLHKPTOpHN0zuRd+P9ln4JzuMgrEbwxS2MH9PIYX8p+3W42nHNJvj8WTame5F/n3NbOH/voK4Baw2bgH6F35yYmJcv7WHAerMRBH7NCDYulmRO+USmj1X7cltoWka/l8RjJ6cpsDzvSRNUyDzz2VboQ/zOy0hJtIpfwrU3R+7dpr4jlKO9Bhn//6W/zFsfeeIirjEGoO+vLwpvstRgPxxVirFI8Z7F/qyF9vWC6DzyxEQqG0kK76TuDVunEzdx5ZpgpAxGXUf78WtUu0tlCq9J0xqEjXlj5ScYrC3X75/6oyvRsE+Opo5ZQF9A9RJM1xIUmBj9JHcgc3LPXbChDMXej7U4+Defc5V4ADlxpfOgurKp3uO8X+wuQ9xuO+ABobsXJMbFkXSs28idBWJOKeHnYeM7Qc2wJeKGXqRwKlZ9dPp8Vq1Tmit6OPvWqusnYzJK8Mre7UTcNlCFvd8tiIfZsdug5YxTOZ7YRa32SUWlz5SQPxCxO+Vv7j9FwTCKJZebGoWxi4AHbQOB/3fQf/3z9L/WV3LMlvdx93u6UH/9ynq/5BRH33gLKAP+X+ctFqp/V/baqH9X7PdOej/PsYfyEznuOIyhjsJayjTuuMEXbp98ez8557A4HXkYACSOPCCNb7Gkw84ER7qDChYHyDQEjAqH0aEw7CTZwnFDeEI7LsjhtBTxEYWZ2y6PsW4Kflk8TnBUlqqFWuZuyTP1KUkLz5ux5bGsHm7e9FTjgk5nDNBmefyjenLHK+PZZB50viJRun98qVNJUbCF1RnwpdxYK8d8qKJmZkqNa/cJDQZkLq6Pxx1VUwXq3guHVnG5D6T+hU/Aw5sFbnbsd6VLy0/ajQ42PsZMp6YalDHFp49ObcxyZt9/urS/uHyxTPM89bA4CfBVZWdpKE0Bxip5HPtZdH/sJnxOnkWLGBWU3Z4fvbq+1cX9tNvL5oX3z412IeFojgMNL631ijWdCTdhh0fVmahDZnB3I4Qjk/Occy8gjoMBB1lrHJBjrfIJUmoYfzV+aMfAt5i7KB+LAMv8AKE5BAv284vhJXBX+Mm9i+TpdJoaRzrAYYy0MgtJqYEYgOZawrjbGe56QqlMPmbF1MEl85wWFzzw+F2+Dv8Hf4Of4e/vX//D/JdY1IAGAEA'
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE)), mode='r:gz').extractall('.')
sys.path.insert(0, W0); os.makedirs('results', exist_ok=True)
from vsearch.weights import discover
t=time.time(); Wt = discover('/kaggle/input', '/tmp/w'); Wt['assemble_s']=round(time.time()-t,1)
R['weights']={k:v for k,v in Wt.items() if k!='layout'}
ok = Wt.get('dinov3') and Wt.get('sam3_ckpt') and Wt.get('sam3_sha256_ok', False)
if not ok: R['errors'].append('weights missing / sha mismatch'); save(); raise SystemExit('stop')
os.environ['DINOV3_PATH']=Wt['dinov3']; os.environ['SAM3_CKPT']=Wt.get('sam3_ckpt') or ''; os.environ['SAM3_STRICT']='1'
run('nvidia_smi','nvidia-smi --query-gpu=name,memory.total --format=csv'); print(R['weights'])

In [ ]:
# EG2 weights: flat 'eg2files__a@b' -> /tmp/eg2/a/b
EG2=None
for p in glob.glob('/kaggle/input/**/eg2files__*', recursive=True):
    rel=os.path.basename(p).split('__',1)[1].replace('@','/'); d=os.path.join('/tmp/eg2',os.path.dirname(rel)); os.makedirs(d,exist_ok=True)
    if not os.path.exists('/tmp/eg2/'+rel): os.symlink(p,'/tmp/eg2/'+rel)
    EG2='/tmp/eg2'
R['eg2_path']=EG2
run('install','pip -q install faiss-cpu pycocotools psutil && git clone -q https://github.com/facebookresearch/sam3.git /tmp/sam3 && pip -q install -e /tmp/sam3 && cd /tmp/sam3 && git rev-parse HEAD', 900)
run('install_eg2','pip -q install --no-deps --target /tmp/eglib "transformers>=5.20" "sentence-transformers>=6.2" "huggingface_hub>=1.0" "tokenizers>=0.22" "safetensors>=0.6" && python -c "import sys;sys.path.insert(0,\'/tmp/eglib\');import transformers,sentence_transformers as s;print(transformers.__version__,s.__version__)"', 900)
os.environ['SAM3_BPE']='/tmp/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz'
print({k:(v['rc'],v['tail'][-200:]) for k,v in R['stages'].items() if k.startswith('install')})

In [ ]:
run('data','wget -q https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip -O /tmp/davis.zip && mkdir -p data && unzip -q /tmp/davis.zip -d data && rm /tmp/davis.zip && python scripts/prepare_davis.py && python scripts/make_hard_queries.py && python scripts/add_distractors.py', 1200)
print(R['stages']['data']['rc'])

In [ ]:
T0=time.time(); BUDGET=100*60
def left(): return BUDGET-(time.time()-T0)
SAN=["import json, glob, sys, os, time, torch; sys.path.insert(0, '.')",
 "from PIL import Image",
 "from vsearch.core import sample_frames",
 "from vsearch.sam3_regions import SAM3Regions",
 "sam = SAM3Regions(); out = {'load_info': sam.load_info, 'precision': os.environ.get('DE12_SAM_DTYPE','fp16'), 'prompts': list(sam.prompts), 'threshold': sam.th}",
 "im = Image.open('/tmp/sam3/assets/images/truck.jpg').convert('RGB')",
 "out['official_truck'] = sam.raw(im, 'truck'); out['official_object'] = sam.raw(im, 'object')",
 "fr = [im for v in sorted(glob.glob('data/davis_bench_large/videos/*.mp4'))[:5] for _, im in list(sample_frames(v, 1.0))[:1]]",
 "t = time.time(); n = [len(sam(f)) for f in fr]; out['davis5_boxes'] = n; out['davis5_s_per_frame'] = round((time.time()-t)/len(fr), 3)",
 "out['ok'] = out['official_truck']['n_ge_th'] > 0 and sum(n) > 0",
 "json.dump(out, open('results/sanity.json', 'w'), indent=1); print(out)"]
open('sanity.py','w').write(chr(10).join(SAN))
R['sam_sanity_tries']={}
for DT in ['fp16','bf16']:
    os.environ['DE12_SAM_DTYPE']=DT; run('sam_sanity_'+DT,'python sanity.py', 900)
    try: R['sam_sanity']=json.load(open('results/sanity.json')); os.remove('results/sanity.json')
    except Exception as e: R['sam_sanity']={'ok':False,'error':repr(e)}
    R['sam_sanity_tries'][DT]=R['sam_sanity']
    if R['sam_sanity'].get('ok'): break
SAM_OK=bool(R['sam_sanity'].get('ok')); save(); print(R['sam_sanity'])
if not SAM_OK: R['errors'].append('SAM sanity failed -> SAM stages skipped (AMG fallback not available for SAM 3 image model in this repo)')
run('gpu_bench','python scripts/gpu_bench.py', 900)
try: R['gpu_bench']=json.load(open('results/gpu_bench.json'))
except Exception as e: R['errors'].append('gpu_bench json: %r'%e)
save(); print(R.get('gpu_bench'))

In [ ]:
R['index']={}; R['eval']={}; R['skipped']=[]
SETS={'large70':'data/davis_bench_large','small10':'data/davis_bench'}; D=os.environ['DINOV3_PATH']
EGENV='PYTHONPATH=/tmp/eglib TRANSFORMERS_VERBOSITY=error'
for sn,B in SETS.items():
    reg='grid+sam3' if SAM_OK else 'grid'
    run(f'index_dinov3_{sn}',f'python scripts/build_index.py --videos {B}/videos --out data/{sn}_d3 --model {D} --regions {reg}', 2400)
    if EG2: run(f'index_eg2_{sn}',f'{EGENV} python scripts/build_index.py --videos {B}/videos --out data/{sn}_eg2 --model {EG2} --regions grid', 1500)
for d in glob.glob('data/*_d3')+glob.glob('data/*_eg2'):
    try: m=json.load(open(d+'/meta.json')); m.pop('meta',None); R['index'][os.path.basename(d)]=m
    except Exception as e: R['errors'].append(f'index meta {d}: {e!r}')
save(); print(R['index'])

In [ ]:
for sn,B in SETS.items():
  for Q in ['object_hard','object']:
    jobs=[('dinov3', f'--index data/{sn}_d3 --modes '+('frame,region,both,sam,frame+sam,fuse' if SAM_OK else 'frame,region,both'), '')]
    if SAM_OK: jobs.append(('dinov3_rerank_sam_top20', f'--index data/{sn}_d3 --modes both --rerank samidx --videos {B}/videos', ''))
    if EG2: jobs.append(('eg2', f'--index data/{sn}_eg2 --modes frame,region,both', EGENV))
    for tag,cmd,env in jobs:
        if not os.path.exists(cmd.split()[1]+'/meta.json'): R['skipped'].append(f'{sn}/{tag}/{Q} (no index)'); continue
        if left()<120: R['skipped'].append(f'{sn}/{tag}/{Q} (time)'); continue
        o=f'results/E_{sn}_{tag}_{Q}.json'; run(f'eval_{sn}_{tag}_{Q}', f'{env} python scripts/evaluate.py {cmd} --queries {B}/queries_{Q}.json --out {o}', max(120,int(left())))
        try: R['eval'][f'{sn}/{tag}/{Q}']=json.load(open(o))['results']
        except Exception as e: R['errors'].append(f'eval {sn}/{tag}/{Q}: {e!r}')
        save()
R['finished']=time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()); save()
print(json.dumps(R['eval'], indent=1)[:4000]); print('errors', R['errors'], 'skipped', R['skipped'])
for f in os.listdir('/kaggle/working'):
    if f!='de12_results.json': subprocess.run(['rm','-rf','/kaggle/working/'+f])
print(os.listdir('/kaggle/working'))